# Notebook 2: Status Tracking and Results Review

This notebook explains how to inspect experiment completion, load outputs, and judge whether results look reasonable.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

output_dir = ROOT / 'outputs' / 'sample_run'
status_path = output_dir / 'status.json'
summary_path = output_dir / 'summary.json'
audit_path = output_dir / 'audit.json'
report_path = output_dir / 'REPORT.md'


## Step 1: Check run status

The run writes `status.json` while executing. This lets you see whether it is still running, how many rows are done, and how many are expected.

In [ ]:
if status_path.exists():
    status = json.loads(status_path.read_text(encoding='utf-8'))
    status
else:
    print('Run status not found yet. Execute `python main.py run --config configs/runtime.experiment.json` first.')

## Step 2: Load summary tables

The reporting layer writes CSV tables you can inspect directly with pandas.

In [ ]:
model_by_depth = pd.read_csv(output_dir / 'model_by_depth.csv') if (output_dir / 'model_by_depth.csv').exists() else None
regime_summary = pd.read_csv(output_dir / 'regime_summary.csv') if (output_dir / 'regime_summary.csv').exists() else None
category_scores = pd.read_csv(output_dir / 'category_scores.csv') if (output_dir / 'category_scores.csv').exists() else None
model_by_depth

## Step 3: Check reasonableness

The `audit.json` file flags obvious issues such as missing baselines, invalid score ranges, or suspicious average improvements under perturbation.

In [ ]:
if audit_path.exists():
    audit = json.loads(audit_path.read_text(encoding='utf-8'))
    audit
else:
    print('Audit not found yet. Run `python main.py audit-results --results outputs/sample_run/results.jsonl --output-dir outputs/sample_run`.')

## Step 4: Visualize degradation

A quick plot of average instruction score by depth is often the first sanity check.

In [ ]:
if model_by_depth is not None:
    pivot = model_by_depth.pivot(index='depth', columns='model_name', values='avg_instruction_level_strict')
    ax = pivot.plot(marker='o', figsize=(8, 4), title='Instruction score by depth')
    ax.set_ylabel('Avg instruction-level strict score')
    ax.set_xlabel('Translation depth')